# Netflix Dataset – Machine Learning Data Quality & Preprocessing Analysis

## Objective
Apply data preprocessing and visualization techniques and understand how poor data quality impacts machine learning models and the ML lifecycle.

This notebook contains:
- Part 1: Data Understanding & Quality Issues
- Part 2: Data Cleaning & Preprocessing
- Part 3: Data Visualization & Interpretation
- Part 4: ML Readiness Check
- Part 5: ML Lifecycle Reflection

## Import Required Libraries
We import libraries required for data manipulation, visualization, and machine learning model building.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sns.set_style('whitegrid')

## Load Dataset
The Netflix dataset is loaded into a pandas dataframe for analysis.

In [ ]:
df = pd.read_csv('netflix_titles.csv')
df.head()

# PART 1 — Data Understanding & Quality Issues
Initial dataset exploration is performed to understand structure, data types, and missing values.

### Inspect Dataset Structure
We check dataset shape, column types, and missing values.

In [ ]:
print('Shape:', df.shape)
print('\nData Types:\n', df.dtypes)
print('\nMissing Values:\n', df.isnull().sum())

### Data Quality Issues Identified
1. Missing director values
2. Missing cast values
3. Missing country values
4. Inconsistent duration format
5. Missing date_added values

# PART 2 — Data Cleaning & Preprocessing
In this section, missing values are handled and features are converted into ML-ready format.

### Handle Missing Values
Categorical missing values are filled. Critical date missing rows are removed.

In [ ]:
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna(df['country'].mode()[0], inplace=True)
df['rating'].fillna(df['rating'].mode()[0], inplace=True)
df = df.dropna(subset=['date_added'])

### Convert Data Formats
Date columns are converted and numeric duration values extracted.

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'])
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

df['duration_num'] = df['duration'].str.extract('(\d+)')
df['duration_num'] = pd.to_numeric(df['duration_num'], errors='coerce')

### Encode Categorical Variables
Categorical variables are converted into numerical values.

In [ ]:
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])
df['rating_encoded'] = le.fit_transform(df['rating'])

### Standardize Numerical Features
Standardization ensures consistent feature scaling for ML models.

In [ ]:
scaler = StandardScaler()
num_cols = ['release_year', 'duration_num', 'year_added']
df[num_cols] = scaler.fit_transform(df[num_cols])

# PART 3 — Data Visualization
Visualizations help identify trends and ML-relevant insights.

### Distribution Plot — Release Year
Shows distribution of content release timeline.

In [ ]:
plt.figure()
sns.histplot(df['release_year'], bins=30, kde=True)
plt.title('Distribution of Release Year')
plt.show()

Interpretation: Most content is recent. ML models may bias toward newer content.

### Categorical Count Plot — Content Type

In [ ]:
plt.figure()
sns.countplot(x='type', data=df)
plt.title('Movies vs TV Shows Count')
plt.show()

Interpretation: Movies dominate dataset → ML prediction bias risk.

### Numerical Comparison Plot — Duration vs Type

In [ ]:
plt.figure()
sns.boxplot(x='type', y='duration_num', data=df)
plt.title('Duration Comparison by Content Type')
plt.show()

Interpretation: Duration behaviour differs for movies and TV shows → Important ML feature.

# PART 4 — ML Readiness Check
Model performance is compared before and after preprocessing.

### Model Before Preprocessing

In [ ]:
df_before = pd.read_csv('netflix_titles.csv')
df_before['type_encoded'] = LabelEncoder().fit_transform(df_before['type'])
X_before = df_before[['release_year']]
y_before = df_before['type_encoded']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_before, y_before, test_size=0.2, random_state=42)

model_b = LogisticRegression()
model_b.fit(X_train_b, y_train_b)
pred_b = model_b.predict(X_test_b)

acc_before = accuracy_score(y_test_b, pred_b)
print('Accuracy Before Preprocessing:', acc_before)

### Model After Preprocessing

In [ ]:
df_after = df.dropna(subset=['release_year','duration_num','year_added'])

X_after = df_after[['release_year','duration_num','year_added']]
y_after = df_after['type_encoded']

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_after, y_after, test_size=0.2, random_state=42)

model_a = LogisticRegression()
model_a.fit(X_train_a, y_train_a)
pred_a = model_a.predict(X_test_a)

acc_after = accuracy_score(y_test_a, pred_a)
print('Accuracy After Preprocessing:', acc_after)

# PART 5 — ML Lifecycle Reflection
Preprocessing ensures reliable ML lifecycle execution.

Without preprocessing:
- Data Preparation → Poor quality features
- Training → Low accuracy
- Evaluation → Misleading metrics
- Deployment → Poor predictions

Example: Duration text values must be converted to numeric for ML compatibility.

# Final Conclusion
Preprocessing improves ML model performance, stability, and reliability across lifecycle stages.